In [2]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
project_root = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "src").exists():
        project_root = candidate
        break

if project_root is None:
    project_root = cwd

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_loader import load_stock_data
from src.validation import validate_stock_data
from src.features import create_features
from src.target import create_target
from src.split import chronological_split

In [3]:
df = load_stock_data("../data/raw/AAPL.csv")

validate_stock_data(df)

df = create_features(df)
df = create_target(df)

feature_columns = [
    "Return_1D",
    "Return_Lag_1",
    "Return_Lag_2",
    "Return_5D",
    "Return_10D",
    "Return_20D",
    "Price_SMA20_Ratio",
    "Price_SMA50_Ratio",
    "Price_EMA20_Ratio",
    "RSI_14",
    "MACD_Norm",
    "MACD_Hist",
    "BB_%B",
    "BB_Width",
    "ATR_%",
    "Volatility_20D",
    "Relative_Volume"
]

df = df.dropna(
    subset=feature_columns + ["Target"]
).reset_index(drop=True)

print("Cleaned shape:", df.shape)

Cleaned shape: (10418, 25)


In [4]:
train_df, val_df, test_df = chronological_split(df)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (7292, 25)
Validation: (1563, 25)
Test: (1563, 25)


In [5]:
for name, part in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(
        name,
        "|",
        part["Date"].min(),
        "to",
        part["Date"].max(),
        "| Rows:",
        len(part)
    )

Train | 1981-02-24 00:00:00 to 2010-01-15 00:00:00 | Rows: 7292
Validation | 2010-01-19 00:00:00 to 2016-04-04 00:00:00 | Rows: 1563
Test | 2016-04-05 00:00:00 to 2022-06-16 00:00:00 | Rows: 1563


In [6]:
for name, part in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"\n{name} target proportions:")
    print(part["Target"].value_counts(normalize=True).sort_index())


Train target proportions:
Target
0.0    0.517828
1.0    0.482172
Name: proportion, dtype: float64

Validation target proportions:
Target
0.0    0.478567
1.0    0.521433
Name: proportion, dtype: float64

Test target proportions:
Target
0.0    0.465771
1.0    0.534229
Name: proportion, dtype: float64
